# 21 — AI Gateway for LLM Backends: Resiliency and Governance

The same API Management instance fronting your MCP servers (notebooks
16-20) can front LLM backends too -- Azure OpenAI, Microsoft Foundry
models, or other providers. This notebook covers the
[AI gateway capabilities](https://learn.microsoft.com/en-us/azure/api-management/genai-gateway-capabilities)
that apply to *chat/completions* traffic rather than MCP tool calls, so you
can combine both behind one gateway in notebook 22.

## Backends, load balancing, and circuit breakers

Configure your LLM endpoints as API Management
[backends](https://learn.microsoft.com/en-us/azure/api-management/backends)
to get:

- **Load balancing** -- round-robin, weighted, priority-based, or
  session-aware distribution across multiple deployments (for example, two
  Azure OpenAI resources in different regions).
- **Circuit breakers** -- API Management stops sending traffic to an
  unhealthy backend and honors the backend's own `Retry-After` header for
  precise, automatic recovery.

Priorities are especially useful when you've purchased Provisioned
Throughput Units (PTUs) on one deployment: route there first, and only
fall back to pay-as-you-go deployments when the PTU backend is saturated or
unhealthy.

## Token governance

Cap token usage per consumer with
[`azure-openai-token-limit`](https://learn.microsoft.com/en-us/azure/api-management/azure-openai-token-limit-policy)
(or the backend-agnostic `llm-token-limit` for non-Azure-OpenAI models):

```xml
<inbound>
    <base />
    <azure-openai-token-limit
        counter-key="@(context.Subscription.Id)"
        tokens-per-minute="10000"
        estimate-prompt-tokens="true"
        remaining-tokens-variable-name="remainingTokens" />
</inbound>
```

This runs *before* the backend call, so a caller that's already over quota
gets a fast 429 instead of an expensive LLM call that's thrown away.

## Metrics and semantic caching

Emit per-consumer token metrics to Application Insights with
[`azure-openai-emit-token-metric`](https://learn.microsoft.com/en-us/azure/api-management/azure-openai-emit-token-metric-policy):

```xml
<outbound>
    <base />
    <azure-openai-emit-token-metric namespace="openai">
        <dimension name="Subscription ID" value="@(context.Subscription.Id)" />
        <dimension name="Client IP" value="@(context.Request.IpAddress)" />
    </azure-openai-emit-token-metric>
</outbound>
```

Reduce repeated LLM calls for semantically similar prompts with
[`azure-openai-semantic-cache-lookup`](https://learn.microsoft.com/en-us/azure/api-management/azure-openai-semantic-cache-lookup-policy)
/ `-store`, backed by an embeddings deployment:

```xml
<inbound>
    <base />
    <azure-openai-semantic-cache-lookup
        score-threshold="0.05"
        embeddings-backend-id="embeddings-backend"
        embeddings-backend-auth="system-assigned" />
</inbound>
<outbound>
    <base />
    <azure-openai-semantic-cache-store duration="60" />
</outbound>
```

Together these turn on the same "log everything, cap everyone, cache
what's safe" governance model you already applied to MCP tools in
notebook 19 -- now for the model calls themselves.

In [ ]:
# Because API Management mirrors the Azure OpenAI REST surface, pointing an
# existing OpenAI SDK client at the gateway instead of the raw resource is
# a one-line change: swap the endpoint, keep the auth pattern from
# src/azure_openai_helper.py.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import AzureOpenAI

from src.config import get_settings

settings = get_settings()
if not settings.apim_gateway_endpoint:
    print(
        "Set APIM_GATEWAY_ENDPOINT in your .env to the API Management path "
        "that fronts your Azure OpenAI deployment to run this cell for real."
    )
else:
    token_provider = get_bearer_token_provider(
        DefaultAzureCredential(), "https://cognitiveservices.azure.com/.default"
    )
    gateway_client = AzureOpenAI(
        azure_endpoint=settings.apim_gateway_endpoint,
        api_version="2024-04-01-preview",
        azure_ad_token_provider=token_provider,
    )
    print("AzureOpenAI client now targets the AI gateway, not the raw resource.")

## Putting MCP and LLM traffic behind one gateway

Nothing stops the same API Management instance from hosting both the MCP
server from notebooks 17/20 *and* the Azure OpenAI backend above. A single
gateway hostname then fronts an agent's entire dependency graph:

```
Agent
  |
  |-- POST /openai-gateway/chat/completions   (Azure OpenAI, load-balanced, cached)
  '-- POST /weather-mcp/mcp                    (MCP tools, rate-limited, traced)
```

One set of Entra ID app registrations, one Application Insights resource,
one place to look when something goes wrong. Notebook 22 wires this up
end to end in Python.

## Recap

| Concern | Policy / feature |
| --- | --- |
| Spread load, tolerate failures | Backends: load balancer + circuit breaker |
| Cap usage per consumer | `azure-openai-token-limit` / `llm-token-limit` |
| Observe usage | `azure-openai-emit-token-metric` / `llm-emit-token-metric` |
| Cut redundant calls | `azure-openai-semantic-cache-lookup` / `-store` |

Continue to
[22_end_to_end_agent_over_apim_gateway.ipynb](22_end_to_end_agent_over_apim_gateway.ipynb)
for the capstone: one agent, calling both the model *and* its tools
through API Management.